# 卷积层

前几章我们逐步构建了一个能处理图像的神经网络：展平层把图像拉成向量，全连接层提取特征，Dropout 防止过拟合，CELoss 高效训练多分类任务。

但这个网络有一个根本性的缺陷——它丢失了图像的**空间结构**。

---

**问题：全连接网络不理解图像**

全连接层把 784 个像素当作 784 个独立特征来处理。这意味着：位置 (5, 3) 和位置 (5, 4) 的像素之间的相邻关系，对网络来说毫无意义——它们只是两个普通的特征，跟不相邻的特征没有区别。

人类识别数字 **7**，是因为看到了"一横加一撇"的形状：一种由相邻像素共同构成的局部结构。全连接网络无法利用这种局部相关性，只能死记硬背每个位置的像素值，泛化能力差。

---

**解决方案：卷积层**

**卷积层**（Convolutional Layer）是专为处理具有网格结构的数据（图像、音频、视频）而设计的神经网络组件。其核心思想是：用一个小型权重数组（**卷积核**，kernel）在输入图像上滑动，提取局部特征。

卷积层有两个关键特性：
- **局部连接**（Local Connectivity）：每个输出值只与输入的一个小区域相关，天然捕捉相邻像素的空间关系
- **权重共享**（Weight Sharing）：同一个卷积核在整张图像上滑动，处处使用相同的参数

权值共享是实现**平移不变性**（translation invariance）的关键：同一个卷积核能在图像的任何位置识别同一种局部特征（比如一条横线、一个角点），无论这个特征出现在左上角还是右下角。这也使得参数量大幅减少。

---

**卷积的工作过程**

以一个 4×4 的输入图像和 2×2 的卷积核为例：

```{figure} images/kernel-1.png
:align: center
:width: 190px
```

卷积核定义（2×2）：

```{figure} images/kernel-2.png
:align: center
:width: 105px
```

卷积核在输入上逐行逐列滑动，每一步与对应区域做加权求和：

```{figure} images/kernel-3.png
:align: center
:width: 640px
```

每步加权求和后得到一个卷积值：

```{figure} images/kernel-4.png
:align: center
:width: 180px
```

所有卷积值合并成输出特征图（3×3）：

```{figure} images/kernel-5.png
:align: center
:width: 120px
```

输出尺寸公式：`输出尺寸 = 输入尺寸 - k + 1`，其中 `k` 为卷积核边长。`4×4` 的输入配 `2×2` 的卷积核，输出为 `3×3`。

---

**多通道与多卷积核**

实际图像有颜色通道（如 RGB 三通道），卷积核也扩展为三维（与输入通道数相同），对所有通道的卷积结果再求和后，得到一个标量输出。

一个卷积层通常也会有**多个卷积核**，每个卷积核采用不同的权重，检测一种不同的局部特征（横边、竖边、斜线……），各自输出一张特征图，合并后构成多通道输出。例如：1×28×28 的 MNIST 图像，经过 16 个 3×3 卷积核的卷积层，输出为 16×26×26。

堆叠多个卷积层后，网络可以从低层的边缘特征，逐步组合成中层的形状，再到高层的整体识别。

In [1]:
from abc import abstractmethod, ABC

import numpy as np

np.random.seed(99)

## 基础架构

### 张量

In [2]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    @property
    def shape(self):
        return self.data.shape

    def __repr__(self):
        return f'Tensor({self.data})'

### 基础数据集

In [3]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size

        self.test_features = None
        self.test_labels = None
        self.train_features = None
        self.train_labels = None

        self.load()

        self.features = self.train_features
        self.labels = self.train_labels

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.features = self.train_features
        self.labels = self.train_labels

    def eval(self):
        self.features = self.test_features
        self.labels = self.test_labels

    def items(self):
        return Tensor(self.features), Tensor(self.labels)

    def __len__(self):
        return len(self.features) // self.batch_size

    def __getitem__(self, index):
        start = index * self.batch_size
        end = start + self.batch_size

        feature = Tensor(self.features[start: end])
        label = Tensor(self.labels[start: end])
        return feature, label

### 基础层

In [4]:
class Layer(ABC):

    def __init__(self):
        self.training = True

    def __call__(self, x: Tensor):
        return self.forward(x)

    def train(self):
        self.training = True

    def eval(self):
        self.training = False

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

    def __repr__(self):
        return f"{type(self).__name__}[]"

### 基础损失函数

In [5]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

### 基础优化器

In [6]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def reset(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    @abstractmethod
    def step(self):
        pass

### 基础模型

In [7]:
class Model(ABC):

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    @abstractmethod
    def train(self, dataset, epochs):
        pass

    @abstractmethod
    def test(self, dataset):
        pass

## 数据

### MNIST 数据集

In [8]:
class MNISTDataset(Dataset):

    def __init__(self, filename, batch_size=1):
        self.filename = filename
        super().__init__(batch_size)

    def load(self):
        with np.load(self.filename, allow_pickle=True) as f:
            self.train_features, self.train_labels = self.preprocess(f['x_train'], f['y_train'])
            self.test_features, self.test_labels = self.preprocess(f['x_test'], f['y_test'])

    @staticmethod
    def preprocess(x, y):
        inputs = x / 255.0
        inputs = np.expand_dims(inputs, axis=1)
        targets = np.zeros((len(y), 10))
        targets[np.arange(len(y)), y] = 1
        return inputs, targets

    def estimate(self, predictions):
        predicted_digits = predictions.data.argmax(axis=1)
        digits = self.labels.argmax(axis=1)
        correct = (predicted_digits == digits).sum()
        return correct / len(self.labels)

## 模型

### 线性层

In [9]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        super().__init__()
        self.weight = Tensor(np.random.rand(out_size, in_size) / in_size)
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

    def __repr__(self):
        return f'Linear[weight{self.weight.shape}; bias{self.bias.shape}]'

### 顺序层

In [10]:
class Sequential(Layer):

    def __init__(self, layers):
        super().__init__()
        self.layers = layers

    def train(self):
        super().train()
        for l in self.layers:
            l.train()

    def eval(self):
        super().eval()
        for l in self.layers:
            l.eval()

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

    def __repr__(self):
        return '\n'.join(str(l) for l in self.layers if str(l))

### 卷积层

**前向传播**：设输入为 $X$（形状：`[batch, channels, height, width]`），一个卷积核权重为 $W$（形状：`[channels, kernal, kernal]`），偏置为 $b$，则输出特征图上位置 $(i, j)$ 的值为：
$$
Y_{i,j} = \sum_{c=0}^{C-1}\sum_{m=0}^{k-1} \sum_{n=0}^{k-1} X_{c,\, i+m,\, j+n} \cdot W_{c,m,n} + b
$$
其中 $C$ 为输入通道数，$k$ 为卷积核边长。输出尺寸为 `height - k + 1` × `width - k + 1`。

**梯度计算**：
- **卷积核权重的梯度**：对每个输出位置的梯度 `p_grad` 与对应输入区块 `patches` 做外积，累加即得权重梯度；
- **偏置的梯度**：对输出梯度在批次和通道维度求和；
- **输入的梯度**（反向传播到上一层）：相当于用输出梯度与权重做**转置卷积**：把每个位置的输出梯度乘以权重矩阵，分散回输入对应的区块位置。

对所有梯度还要除以批大小 `batch`，这是梯度按批归一化的惯例，使梯度不随批大小变化而缩放。

**父节点集合**（parents）：输入值。

**参数列表**（parameters）：每个卷积核的权重和偏置。

---

**实现技巧：kaiming 随机初始化**

和我们在线性层使用的**均匀分布随机初始化**不同，我们在卷积层使用 **kaiming 随机初始化**，这是配合 ReLU 激活函数的推荐初始化方式，能使各层输出的方差保持稳定。公式为：

$$
\sqrt{\frac{2}{channels \cdot k^2}}
$$

---

**实现技巧：im2col 图转列**

直接的卷积实现需要嵌套多层循环，效率极低。实践中通常用 **im2col** 技术：

* 把每个卷积步骤覆盖的输入区域展平成一行，所有步骤的行拼接成一个大矩阵；
* 再把卷积核也展平；
* 通过矩阵乘法一次完成所有步骤的计算。

这是以空间换时间，显著提升 NumPy 向量化运算的效率。

In [11]:
class Convolution(Layer):

    def __init__(self, in_channels: int, out_channels: int, kernel_size: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        # kaiming 随机初始化
        self.weight = Tensor(np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * np.sqrt(2.0 / (in_channels * kernel_size ** 2)))
        self.bias = Tensor(np.zeros(out_channels))

    def forward(self, x: Tensor) -> Tensor:
        batch, in_ch, in_h, in_w = x.data.shape
        # 输出尺寸
        out_h = in_h - self.kernel_size + 1
        out_w = in_w - self.kernel_size + 1

        # im2col 图转列
        # 输出形状：每个输出位置扩展为一个卷积核（kernel_size, kernel_size）
        shape = batch, in_ch, out_h, out_w, self.kernel_size, self.kernel_size
        # 各维度的步长
        strides = x.data.strides[0], x.data.strides[1], x.data.strides[2], x.data.strides[3], x.data.strides[2], x.data.strides[3]
        # 根据输出形状和步长创建新的视图（不需要移动数据）
        patches = np.lib.stride_tricks.as_strided(x.data, shape=shape, strides=strides)
        # 重排维度并展平，得到 im2col 矩阵：（batch, out_height * out_width, in_channels * kernal_height * kernal_width）
        patches = patches.transpose(0, 2, 3, 1, 4, 5).reshape(batch, out_h * out_w, -1)

        # 展平权重（out_channels, in_channels * kernal_height * kernal_width）
        weight = self.weight.data.reshape(self.out_channels, -1)
        # 加权求和（batch, out_height * out_width, out_channels）
        output = (patches @ weight.T + self.bias.data)
        # 还原为标准特征图格式（batch, out_channels, out_height, out_width）
        output = output.reshape(batch, out_h, out_w, self.out_channels).transpose(0, 3, 1, 2)
        p = Tensor(output)

        def gradient_fn():
            # 转置回 (batch, out_height * out_width, out_channels) ，以匹配前向的矩阵形状
            p_grad = p.grad.transpose(0, 2, 3, 1).reshape(batch, out_h * out_w, self.out_channels)
            # 计算权重，偏置的梯度
            w_grad = p_grad.reshape(-1, self.out_channels).T @ patches.reshape(-1, patches.shape[-1])
            self.weight.grad += w_grad.reshape(self.weight.data.shape) / batch
            self.bias.grad += p_grad.sum(axis=(0, 1)) / batch

            # im2col 的逆操作：
            # 计算输入值的梯度，并重排回 im2col 矩阵的形状
            p_grad_patches = (p_grad @ weight).reshape(batch, out_h, out_w, in_ch, self.kernel_size, self.kernel_size)

            # 卷积核的位置索引
            out_h_index = np.arange(out_h)[:, None] + np.arange(self.kernel_size)[None, :]
            out_w_index = np.arange(out_w)[:, None] + np.arange(self.kernel_size)[None, :]
            # 各维度的位置索引
            batch_index = np.arange(batch)[:, None, None, None, None, None]
            ch_index = np.arange(in_ch)[None, None, None, :, None, None]
            h_index = out_h_index[None, :, None, None, :, None]
            w_index = out_w_index[None, None, :, None, None, :]
            # 累加计算输入值各位置的梯度
            grad = np.zeros_like(x.data, dtype=float)
            np.add.at(grad, (batch_index, ch_index, h_index, w_index), p_grad_patches)
            x.grad += grad / batch

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self) -> list[Tensor]:
        return [self.weight, self.bias]

    def __repr__(self) -> str:
        return f"Convolution[weight={self.weight.shape}; bias={self.bias.shape}; kernels={self.out_channels}; kernel_size={self.kernel_size}]"

### 展平层

In [12]:
class Flatten(Layer):

    def forward(self, x: Tensor):
        p = Tensor(np.array(x.data.reshape(x.data.shape[0], -1)))

        def gradient_fn():
            x.grad += p.grad.reshape(x.data.shape)

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

### 丢弃层

In [13]:
class Dropout(Layer):

    def __init__(self, dropout_rate=0.2):
        super().__init__()
        self.dropout_rate = dropout_rate

    def forward(self, x: Tensor):
        if not self.training:
            return x

        mask = np.random.random(x.data.shape) > self.dropout_rate
        p = Tensor(x.data * mask)

        def gradient_fn():
            x.grad += p.grad * mask

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    def __repr__(self):
        return f'Dropout[rate={self.dropout_rate}]'

### ReLU 激活函数层

In [14]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Tanh 激活函数层

In [15]:
class Tanh(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.tanh(x.data))

        def gradient_fn():
            x.grad += a.grad * (1 - a.data ** 2)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Sigmoid 激活函数层

In [16]:
class Sigmoid(Layer):

    def __init__(self, clip_range=(-100, 100)):
        super().__init__()
        self.clip_range = clip_range

    def forward(self, x: Tensor):
        z = np.clip(x.data, self.clip_range[0], self.clip_range[1])
        a = Tensor(1 / (1 + np.exp(-z)))

        def gradient_fn():
            x.grad += a.grad * a.data * (1 - a.data)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### Softmax 激活函数层

In [17]:
class Softmax(Layer):

    def __init__(self, axis=-1):
        super().__init__()
        self.axis = axis

    def forward(self, x: Tensor):
        exp = np.exp(x.data - np.max(x.data, axis=self.axis, keepdims=True))
        a = Tensor(exp / np.sum(exp, axis=self.axis, keepdims=True))

        def gradient_fn():
            grad = np.sum(a.data * a.grad, axis=self.axis, keepdims=True)
            x.grad += a.data * (a.grad - grad)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

### 均方误差损失函数

In [18]:
class MSELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / len(y.data)

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

### 交叉熵损失函数

In [19]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        log = np.log(np.clip(softmax, 1e-10, 1))
        ce = Tensor(0 - np.sum(y.data * log) / len(y.data))

        def gradient_fn():
            p.grad += (softmax - y.data) / len(y.data)

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

### 二元交叉熵损失函数

In [20]:
class BCELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        clipped = np.clip(p.data, 1e-7, 1 - 1e-7)
        bce = Tensor(-np.mean(y.data * np.log(clipped) + (1 - y.data) * np.log(1 - clipped)))

        def gradient_fn():
            p.grad += (clipped - y.data) / (clipped * (1 - clipped)) / len(y.data)

        bce.gradient_fn = gradient_fn
        bce.parents = {p}
        return bce

### 随机梯度下降优化器

In [21]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

### 神经网络模型

In [22]:
class NNModel(Model):

    def train(self, dataset, epochs):
        self.layer.train()
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                features, labels = dataset[i]

                self.optimizer.reset()
                predictions = self.layer(features)
                loss = self.loss_fn(predictions, labels)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        self.layer.eval()
        dataset.eval()

        features, labels = dataset.items()
        predictions = self.layer(features)
        loss = self.loss_fn(predictions, labels)
        return predictions, loss

## 训练

### 超参数：学习率

In [23]:
LEARNING_RATE = 0.01

### 超参数：批大小

In [24]:
BATCH_SIZE = 2

### 超参数：卷积核大小

In [25]:
KERNEL_SIZE = 3

### 超参数：轮次

In [26]:
EPOCHS = 10

### 建模

把卷积层加到网络的第一层，让它先提取图像的局部特征，再交给全连接层分类。

卷积层输出的特征图形状是 `[batch, 16, 26, 26]`（16 个通道，每通道 26×26），需要先经过展平层，再接线性层。线性层的输入维度为 16×26×26。

In [27]:
dataset = MNISTDataset('tinymnist.npz', BATCH_SIZE)

layer = Sequential([Convolution(1, 16, KERNEL_SIZE),
                    Flatten(),
                    Dropout(),
                    Linear(16 * (28 - KERNEL_SIZE + 1) ** 2, 64),
                    ReLU(),
                    Linear(64, 10)])
loss_fn = CELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)

model = NNModel(layer, loss_fn, optimizer)
print(layer)

Convolution[weight=(16, 1, 3, 3); bias=(16,); kernels=16; kernel_size=3]
Flatten[]
Dropout[rate=0.2]
Linear[weight(64, 10816); bias(64,)]
ReLU[]
Linear[weight(10, 64); bias(10,)]


### 迭代

In [28]:
model.train(dataset, EPOCHS)

## 验证

### 测试

In [29]:
predictions, loss = model.test(dataset)
accuracy = dataset.estimate(predictions)
print(f'accuracy: {accuracy:.2%}')

accuracy: 93.30%


卷积层带来了明显的准确率提升，这正是卷积层的威力：它保留了图像的空间结构，使网络能识别局部形状特征，而不只是死记单个像素的位置。

但随之而来的是速度明显变慢，同时打印出来的线性层参数量也大幅增加。让我们算一下：

* 卷积层输出形状：16×26×26 = **10,816 个特征**；
* 第一个线性层：10,816 → 64，共 10,816×64 + 64 = **692,288 个参数**。

这对于一个如此简单的任务来说完全没必要，而且参数越多，越难训练、越容易过拟合。这就是**参数爆炸**问题。卷积层压缩了模型的卷积核参数，但后续线性层的参数却膨胀了。

## 课后练习

在卷积层后紧接一个 ReLU 激活函数层，再加第二个卷积层（输入通道 16，输出通道 32），观察第二个卷积层的输出尺寸变化，并更新后续线性层的输入维度。